# arange-fancy-index-cross-entropy — worked example 2: Sum-reduced cross-entropy via logsumexp minus picked

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `arange-fancy-index-cross-entropy`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

Per-sample cross-entropy decomposes as `ce_i = logsumexp(logits_i) - logits[i, target[i]]`. The second term is the canonical `arange` fancy-index gather. Summing over the batch matches `F.cross_entropy(..., reduction='sum')`.

## Worked solution

**Goal.** Return a scalar equal to `F.cross_entropy(logits, target, reduction='sum')`, built from primitives.

**Step 1 — logsumexp per row.** `lse = t.logsumexp(logits, dim=-1)` gives a `(B,)` vector, the log-normalizer of each row's softmax. This is the `-log` of the denominator in softmax, computed stably.

**Step 2 — pick the target logit.** `picked = logits[t.arange(B), target]` gathers the raw logit at the true class for each row, shape `(B,)`. The `arange(B)` row index zips positionally with `target`.

**Step 3 — per-sample CE.** `ce = lse - picked`, shape `(B,)`. This is correct because `-log softmax(logits)[i, c] = logsumexp_c'(logits_i) - logits[i, c]`.

**Step 4 — reduce by sum.** `ce.sum()` collapses to a 0-D scalar. Using `sum` rather than `mean` matches `reduction='sum'`. The result has `.shape == ()`.

In [ ]:
def ce_sum(logits, target):
    B = logits.shape[0]
    lse = t.logsumexp(logits, dim=-1)             # (B,)
    picked = logits[t.arange(B), target]          # (B,)
    return (lse - picked).sum()                   # scalar

t.manual_seed(0)
logits = t.randn(6, 3)
target = t.tensor([0, 2, 1, 1, 0, 2])
loss = ce_sum(logits, target)
ref = t.nn.functional.cross_entropy(logits, target, reduction='sum')
print(loss.shape)
print(float(loss), float(ref))